# Analyse: Angereicherte Shot Events (Kinexon)

**Kontext:** Kinexon liefert ab 20. April 2026 angereicherte Shot Events über die bekannte API.
Vorab wurden 4 Testdateien bereitgestellt. Dieses Notebook vergleicht das neue Format
mit den bestehenden Daten in `data/hbl_raw.duckdb` und identifiziert Fragen für den Termin am 14. April.

**Neue Dateien (`.tmp_data/`):**
- `62089852` – Bergischer HC vs SG Flensburg-Handewitt (21.09.2025)
- `62089910` – TBV Lemgo Lippe vs Rhein-Neckar Löwen (16.10.2025)
- `62089952` – TBV Lemgo Lippe vs TSV Hannover-Burgdorf (09.11.2025)
- `62090222` – SC Magdeburg vs Füchse Berlin (28.03.2026) ← noch nicht in DB


In [ ]:
import json
import glob
import os
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyArrowPatch

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)

In [ ]:
# --- Daten laden ---

# Neue enriched JSON-Dateien
DATA_DIR = '../.tmp_data'
DB_PATH = '../data/hbl_raw.duckdb'

files = sorted(glob.glob(os.path.join(DATA_DIR, '*.json')))
print(f'Gefundene Dateien: {len(files)}')
for f in files:
    print(f'  {os.path.basename(f)}')

In [ ]:
# Alle Shots in einen DataFrame laden
all_shots = []
for fpath in files:
    with open(fpath) as f:
        shots = json.load(f)
    # player_positions separat speichern, Rest als flat row
    for shot in shots:
        row = {k: v for k, v in shot.items() if k != 'player_positions'}
        row['n_player_positions'] = len(shot['player_positions'])
        row['has_positions'] = len(shot['player_positions']) > 0
        all_shots.append(row)

df = pd.DataFrame(all_shots)
df['match_id'] = df['match_id'].astype(str)

print(f'Gesamt Shots: {len(df)}')
print(f'Spalten: {list(df.columns)}')

In [ ]:
# player_positions als eigenen DataFrame
pos_rows = []
for fpath in files:
    with open(fpath) as f:
        shots = json.load(f)
    for shot in shots:
        for p in shot['player_positions']:
            pos_rows.append({
                'shot_id': shot['id'],
                'match_id': shot['match_id'],
                'shooter_id': shot['player_id'],
                'goalkeeper_id': shot.get('goalkeeper_id'),
                **p
            })

df_pos = pd.DataFrame(pos_rows)
df_pos['is_shooter'] = df_pos['player_id'] == df_pos['shooter_id']
df_pos['is_goalkeeper'] = df_pos['player_id'] == df_pos['goalkeeper_id']
print(f'Gesamt Spielerpositionen: {len(df_pos)}')
df_pos.head(3)

## 1. Überblick: Coverage und Shot-Statistiken

In [ ]:
# Mapping: JSON match_id (Sportradar) → DB fixture_id (UUID)
# Abgeglichen über Team-Namen + Datum
MATCH_ID_TO_FIXTURE_ID = {
    '62089852': 'f2aab62a-5ca5-11f0-84ec-cb70ad9a9de9',  # Bergischer HC vs Flensburg
    '62089910': '068c8661-5ca6-11f0-9890-6768df442659',  # Lemgo vs RNL
    '62089952': '16c1d65e-5ca6-11f0-9209-116dfc6d7ae4',  # Lemgo vs Hannover
    '62090222': None,  # Magdeburg vs Füchse — noch nicht in DB!
}

MATCH_NAMES = {
    '62089852': 'Bergischer HC vs Flensburg (21.09)',
    '62089910': 'Lemgo vs RNL (16.10)',
    '62089952': 'Lemgo vs Hannover (09.11)',
    '62090222': 'Magdeburg vs Füchse (28.03) ⚠️',
}

ARENA_FORMULA = {
    '62089852': 'B',  # Bergischer HC (Heim) — y-Achse negiert
    '62089910': 'A',  # Lemgo (Heim)
    '62089952': 'A',  # Lemgo (Heim)
    '62090222': 'B',  # Magdeburg (Heim) — y-Achse negiert
}

summary = df.groupby('match_id').agg(
    match_name=('match_id', lambda x: MATCH_NAMES[x.iloc[0]]),
    total_shots=('id', 'count'),
    with_positions=('has_positions', 'sum'),
    validated=('validated', 'sum'),
    goals=('success', 'sum'),
).reset_index()
summary['positions_pct'] = (summary['with_positions'] / summary['total_shots'] * 100).round(1)
summary['goal_rate_validated'] = (summary['goals'] / summary['validated'] * 100).round(1)
print(summary.to_string(index=False))

In [ ]:
# Validated vs. nicht-validated
print('validated=0 (Fehldetektionen):', len(df[df['validated'] == 0]))
print('validated=1 (echte Schüsse):  ', len(df[df['validated'] == 1]))
print()
print('Bei validated=0: Anteil ohne player_positions:')
v0 = df[df['validated'] == 0]
print(f'  {(~v0["has_positions"]).sum()} von {len(v0)} ({(~v0["has_positions"]).mean()*100:.0f}%)')

In [ ]:
# Spieler pro Shot
with_pos = df[df['has_positions']]
print(f'Spielerpositionen pro Shot (n={len(with_pos)})')
print(with_pos['n_player_positions'].describe())

fig, ax = plt.subplots(figsize=(6, 3))
with_pos['n_player_positions'].hist(bins=range(10, 18), ax=ax, edgecolor='black')
ax.set_xlabel('Anzahl Spieler in player_positions')
ax.set_ylabel('Anzahl Shots')
ax.set_title('Spieler pro Shot-Snapshot')
plt.tight_layout()
plt.show()

## 2. Vergleich mit bestehender DB: match_detected_shots_normalized

In [ ]:
con = duckdb.connect(DB_PATH, read_only=True)

# Bestehende detected shots aus DB für die 3 bekannten Fixtures
known_fixture_ids = [fid for fid in MATCH_ID_TO_FIXTURE_ID.values() if fid is not None]
placeholders = ', '.join([f"'{fid}'" for fid in known_fixture_ids])

df_db = con.execute(f"""
    SELECT 
        fixture_id,
        id,
        timestamp_ms,
        game_clock,
        player_id,
        league_id,
        shot_position_x,
        shot_position_y,
        distance,
        speed_ball,
        success,
        shot_category,
        shot_type,
        validated,
        goalkeeper_id
    FROM main.match_detected_shots_normalized
    WHERE fixture_id IN ({placeholders})
""").df()

# fixture_id → SR match_id rückauflösen
inv_map = {v: k for k, v in MATCH_ID_TO_FIXTURE_ID.items() if v}
df_db['match_id'] = df_db['fixture_id'].map(inv_map)
df_db['id'] = df_db['id'].astype(int)

print(f'DB detected shots (3 Fixtures): {len(df_db)}')
df_db.groupby('match_id').size().rename('db_shots')

In [ ]:
# Shot-Counts vergleichen: Neu vs. DB (nur die 3 bekannten Fixtures)
df_known = df[df['match_id'].isin(inv_map.values())].copy()

new_counts = df_known.groupby('match_id').agg(
    new_total=('id', 'count'),
    new_validated=('validated', 'sum'),
    new_with_positions=('has_positions', 'sum'),
)
db_counts = df_db.groupby('match_id').agg(
    db_total=('id', 'count'),
    db_validated=('validated', 'sum'),
)

comparison = new_counts.join(db_counts)
comparison['match_name'] = comparison.index.map(MATCH_NAMES)
comparison['diff_total'] = comparison['new_total'] - comparison['db_total']
comparison['diff_validated'] = comparison['new_validated'] - comparison['db_validated']
print(comparison[['match_name', 'new_total', 'db_total', 'diff_total', 'new_validated', 'db_validated', 'diff_validated']].to_string())

In [ ]:
# Shot-ID-Abgleich: Welche IDs sind in beiden Datensätzen?
results = []
for match_id, fid in MATCH_ID_TO_FIXTURE_ID.items():
    if fid is None:
        continue
    new_ids = set(df[df['match_id'] == match_id]['id'].astype(int))
    db_ids = set(df_db[df_db['match_id'] == match_id]['id'].astype(int))
    only_new = new_ids - db_ids
    only_db = db_ids - new_ids
    both = new_ids & db_ids
    results.append({
        'match': MATCH_NAMES[match_id],
        'in_both': len(both),
        'only_in_new': len(only_new),
        'only_in_db': len(only_db),
    })
    
pd.DataFrame(results)

## 3. ID-System: Wie verbinden sich neue Daten mit bestehenden?

In [ ]:
# Spieler-ID-Systematik klären
# In new format:
#   player_id (top-level) = KNX-interner ID (kleine Integer)
#   league_id (top-level) = Sportradar Player-ID
# In player_positions:
#   player_id = KNX-interner ID
#   league_id = Sportradar Player-ID

print('Neue Daten – ID-Felder:')
print(df[['player_id', 'league_id', 'goalkeeper_id', 'goalkeeper_league_id']].head(5).to_string())
print()
print('player_positions – ID-Felder:')
print(df_pos[['player_id', 'league_id', 'is_shooter', 'is_goalkeeper']].head(8).to_string())

In [ ]:
# Verifizieren: Schütze aus player_positions league_id == top-level league_id?
all_shots_raw = []
for fpath in files:
    with open(fpath) as f:
        all_shots_raw.extend(json.load(f))

shots_with_pos = [s for s in all_shots_raw if s['player_positions']]
shooter_lid_match = sum(
    1 for s in shots_with_pos
    if any(p['player_id'] == s['player_id'] and str(p['league_id']) == str(s['league_id'])
           for p in s['player_positions'])
)
gk_shots = [s for s in shots_with_pos if s.get('goalkeeper_id') is not None]
gk_lid_match = sum(
    1 for s in gk_shots
    if any(p['player_id'] == s['goalkeeper_id'] and str(p['league_id']) == str(s.get('goalkeeper_league_id', ''))
           for p in s['player_positions'])
)

print(f'Schütze: league_id top-level == league_id in player_positions: {shooter_lid_match}/{len(shots_with_pos)} ({100*shooter_lid_match/len(shots_with_pos):.1f}%)')
print(f'Torwart: goalkeeper_league_id == league_id in player_positions: {gk_lid_match}/{len(gk_shots)} ({100*gk_lid_match/len(gk_shots):.1f}%)')
print()
print('→ league_id ist der Sportradar Player-ID (unser Verbindungs-Schlüssel zu bestehenden Daten)')

In [ ]:
# Spieler aus player_positions mit players-Tabelle in DB abgleichen
df_players_db = con.execute("""
    SELECT DISTINCT league_id, name, position, team_name
    FROM main.players
    WHERE league_id IS NOT NULL
""").df()

# Alle distinct league_ids aus den neuen Daten
new_league_ids = set(df_pos['league_id'].astype(str).unique())
db_league_ids = set(df_players_db['league_id'].astype(str).unique())

matched = new_league_ids & db_league_ids
only_new = new_league_ids - db_league_ids

print(f'Distinct player league_ids in neuen Daten:  {len(new_league_ids)}')
print(f'Davon in players-Tabelle DB gefunden:        {len(matched)} ({100*len(matched)/len(new_league_ids):.1f}%)')
print(f'Nur in neuen Daten (unbekannte Spieler):     {len(only_new)}')
if only_new:
    print(f'  Beispiel league_ids: {list(only_new)[:10]}')

## 4. Koordinatensysteme

In [ ]:
# Zwei Koordinatensysteme:
#   player_positions x/y: Absolut, 0-40m × 0-20m (Handballfeld)
#   shot_position_x/y:    Relativ – Herkunft klären

df_valid = df[(df['validated'] == 1) & df['has_positions'] & df['shot_position_x'].notna()].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: shot_position_x/y (das bisherige System) ---
ax = axes[0]
ax.scatter(df_valid[df_valid['success']==0]['shot_position_x'],
           df_valid[df_valid['success']==0]['shot_position_y'],
           c='steelblue', alpha=0.4, s=15, label='Kein Tor')
ax.scatter(df_valid[df_valid['success']==1]['shot_position_x'],
           df_valid[df_valid['success']==1]['shot_position_y'],
           c='red', alpha=0.6, s=20, label='Tor', zorder=3)
# Tor andeuten: x≈0, y∈[-1.5, 1.5]
ax.axvline(0, color='gray', linewidth=0.8, linestyle='--', label='x=0 (Torkante?)')
ax.axhline(0, color='lightgray', linewidth=0.5)
ax.set_xlim(-25, 25)
ax.set_ylim(-12, 12)
ax.set_xlabel('shot_position_x [m]')
ax.set_ylabel('shot_position_y [m]')
ax.set_title('shot_position (zentriertes System)')
ax.legend(fontsize=8)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# --- Plot 2: player_positions absolutes System ---
ax2 = axes[1]
# Feld zeichnen (40×20m)
field = patches.Rectangle((0, 0), 40, 20, linewidth=1.5, edgecolor='green', facecolor='#f0fff0')
ax2.add_patch(field)
ax2.axvline(20, color='green', linewidth=0.8, linestyle='--', alpha=0.5)  # Mittellinie
# Tore andeuten (x=0 und x=40, y∈[8.5, 11.5])
for tx in [0, 40]:
    goal = patches.Rectangle((tx-0.5, 8.5), 0.5, 3, linewidth=1, edgecolor='black', facecolor='white')
    ax2.add_patch(goal)

# Sample Shot: alle Spieler + Schütze markieren
sample_shot = next(s for s in shots_with_pos if s['shot_position_x'] is not None and s['success'] == 1 and len(s['player_positions']) >= 13)
for p in sample_shot['player_positions']:
    color = 'red' if p['player_id'] == sample_shot['player_id'] else \
            'orange' if p['player_id'] == sample_shot.get('goalkeeper_id') else 'blue'
    marker = '*' if p['player_id'] == sample_shot['player_id'] else \
             's' if p['player_id'] == sample_shot.get('goalkeeper_id') else 'o'
    size = 120 if p['player_id'] in [sample_shot['player_id'], sample_shot.get('goalkeeper_id')] else 40
    ax2.scatter(p['x'], p['y'], c=color, s=size, marker=marker, zorder=3)

ax2.set_xlim(-1, 41)
ax2.set_ylim(-1, 21)
ax2.set_xlabel('x [m] (absolut)')
ax2.set_ylabel('y [m] (absolut)')
ax2.set_title('player_positions (absolutes Feldkoordinatensystem)\nBeispiel: 1 Tor-Shot ★=Schütze ■=Torwart')
ax2.set_aspect('equal')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Beziehung shot_position_x/y ↔ player_positions (Schütze)
rows_with_both = []
for s in shots_with_pos:
    if s['shot_position_x'] is None:
        continue
    shooter_pos = next((p for p in s['player_positions'] if p['player_id'] == s['player_id']), None)
    if shooter_pos:
        rows_with_both.append({
            'match_id': str(s['match_id']),
            'shot_pos_x': s['shot_position_x'],
            'shot_pos_y': s['shot_position_y'],
            'player_x': shooter_pos['x'],
            'player_y': shooter_pos['y'],
            'diff_x': shooter_pos['x'] - s['shot_position_x'],
        })

df_coord = pd.DataFrame(rows_with_both)
print('X-Offset (player_x - shot_position_x) — gesamt:')
print(df_coord['diff_x'].describe().round(4))
print()
print('→ Sehr konsistent ~20m: shot_position_x = player_x - 20 (Mittellinie als Nullpunkt)')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(df_coord['diff_x'], bins=30, edgecolor='black')
ax.axvline(20, color='red', linestyle='--', label='x=20 (Feldmitte)')
ax.set_xlabel('player_x - shot_position_x [m]')
ax.set_ylabel('Häufigkeit')
ax.set_title('X-Offset: konsistent ~20m für alle Arenen')
ax.legend()
plt.tight_layout()
plt.show()

### shot_position_y: Arena-abhängiges Vorzeichen

In [ ]:
# Hypothese geprüft: shot_position_y ist ±(player_y - 10) — Vorzeichen arena-abhängig
#
# Formel A:  shot_pos_y =  player_y - 10   (Lemgo)
# Formel B:  shot_pos_y = -(player_y - 10) (Bergischer HC, Magdeburg)
#
# Test: Residuals bei korrekter Formel sollten nahe 0 sein.

results = []
for s in shots_with_pos:
    if s['shot_position_y'] is None or s['validated'] != 1:
        continue
    shooter = next((p for p in s['player_positions'] if p['player_id'] == s['player_id']), None)
    if not shooter:
        continue
    mid = MATCH_NAMES[str(s['match_id'])]
    formula = ARENA_FORMULA[str(s['match_id'])]
    if formula == 'A':
        predicted = shooter['y'] - 10
    else:
        predicted = -(shooter['y'] - 10)
    results.append({
        'match': mid,
        'formula': formula,
        'residual': s['shot_position_y'] - predicted,
    })

df_res = pd.DataFrame(results)
print('Residual (shot_pos_y - predicted) nach Formel:')
print(df_res.groupby(['match', 'formula'])['residual'].agg(['mean', 'std', lambda x: x.abs().max()]).rename(columns={'<lambda_0>': 'max_abs'}).round(4))
print()
print('→ Formel A (player_y - 10):  Lemgo-Arenen')
print('→ Formel B (-(player_y-10)): Bergischer HC, Magdeburg — y-Achse gespiegelt')

In [ ]:
# Visualisierung: shot_pos_y vs. (player_y - 10) pro Arena
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, (match_id, name) in zip(axes, MATCH_NAMES.items()):
    formula = ARENA_FORMULA[match_id]
    sub = [(s['shot_position_y'], shooter['y'] - 10)
           for s in shots_with_pos
           if str(s['match_id']) == match_id and s['shot_position_y'] is not None and s['validated'] == 1
           for shooter in [next((p for p in s['player_positions'] if p['player_id'] == s['player_id']), None)]
           if shooter]
    if not sub:
        continue
    ys, xs = zip(*sub)
    ax.scatter(xs, ys, s=10, alpha=0.5)
    lim = max(abs(min(xs+ys)), abs(max(xs+ys))) + 0.5
    if formula == 'A':
        ax.plot([-lim, lim], [-lim, lim], 'r-', linewidth=1.5, label='y=x (Formel A)')
    else:
        ax.plot([-lim, lim], [lim, -lim], 'g-', linewidth=1.5, label='y=-x (Formel B)')
    ax.set_xlabel('player_y - 10')
    ax.set_ylabel('shot_pos_y')
    ax.set_title(f'{name}\n(Formel {formula})', fontsize=8)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')

plt.suptitle('shot_position_y = ±(player_y − 10): Vorzeichen ist arena-abhängig', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Verteilung shot_position_x und shot_position_y pro Arena
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col, title in zip(axes,
    ['shot_position_x', 'shot_position_y'],
    ['shot_position_x (alle Arenen konsistent)', 'shot_position_y (Vorzeichen arena-abhängig)']):

    df_valid_v = df[(df['validated'] == 1) & df['has_positions'] & df['shot_position_x'].notna()].copy()
    df_valid_v['match_name'] = df_valid_v['match_id'].map(MATCH_NAMES)

    for jitter_i, (name, grp) in enumerate(df_valid_v.groupby('match_name')):
        goals = grp[grp['success'] == 1][col]
        no_goals = grp[grp['success'] == 0][col]
        ax.scatter(no_goals, [jitter_i] * len(no_goals), c='steelblue', alpha=0.3, s=8)
        ax.scatter(goals, [jitter_i] * len(goals), c='red', alpha=0.5, s=12)

    ax.set_yticks(range(len(MATCH_NAMES)))
    ax.set_yticklabels(list(MATCH_NAMES.values()), fontsize=8)
    ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')
    ax.set_xlabel(col)
    ax.set_title(title, fontsize=9)
    ax.grid(True, alpha=0.3, axis='x')

# Legend
from matplotlib.lines import Line2D
handles = [Line2D([0],[0], marker='o', color='w', markerfacecolor='steelblue', label='Kein Tor', markersize=6),
           Line2D([0],[0], marker='o', color='w', markerfacecolor='red', label='Tor', markersize=6)]
axes[1].legend(handles=handles, fontsize=8, loc='upper right')

plt.suptitle('Verteilung nach Arena — x konsistent, y gespiegelt', fontsize=10)
plt.tight_layout()
plt.show()

print()
print('Empfehlung: abs(player_y - 10) als robuste laterale Distanz — funktioniert ohne Arena-Korrektur.')

## 5. Kategorische Felder: shot_type, last_group, shot_category

In [ ]:
df_v = df[df['validated'] == 1].copy()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col, title in zip(axes,
    ['shot_type', 'last_group', 'shot_category'],
    ['shot_type (Encoding unklar)', 'last_group (Bedeutung unklar)', 'shot_category']):
    
    vc = df_v[col].value_counts(dropna=False).sort_index()
    goal_rates = df_v.groupby(col, dropna=False)['success'].mean()
    
    bars = ax.bar([str(v) for v in vc.index], vc.values, color='steelblue', alpha=0.7)
    ax2 = ax.twinx()
    ax2.plot([str(v) for v in vc.index], [goal_rates.get(k, 0) * 100 for k in vc.index],
             'ro-', markersize=6, label='Torquote %')
    ax.set_title(title)
    ax.set_xlabel(col)
    ax.set_ylabel('Anzahl Shots')
    ax2.set_ylabel('Torquote [%]', color='red')
    ax2.tick_params(axis='y', labelcolor='red')

plt.tight_layout()
plt.show()

print('last_group=3 Shots (alle validated=1):')
print(df_v[df_v['last_group'] == 3][['match_id', 'game_clock', 'success', 'shot_category', 'shot_type']].to_string())

## 6. player_positions: Spielerverteilung beim Schuss

In [ ]:
# Abstand des nächsten Gegenspielers zum Schützen
# Dafür: Schützteam identifizieren (alle aus player_positions haben keine team-Info!)
# Nur Schützen-Position bekannt → Abstand zu allen anderen Spielern

dist_rows = []
for s in shots_with_pos:
    if s['shot_position_x'] is None or s['validated'] != 1:
        continue
    shooter = next((p for p in s['player_positions'] if p['player_id'] == s['player_id']), None)
    gk = next((p for p in s['player_positions']
                if p['player_id'] == s.get('goalkeeper_id')), None)
    if not shooter:
        continue
    
    others = [p for p in s['player_positions']
              if p['player_id'] != s['player_id']
              and p['player_id'] != s.get('goalkeeper_id')]
    
    dists = [np.sqrt((p['x'] - shooter['x'])**2 + (p['y'] - shooter['y'])**2) for p in others]
    gk_dist = np.sqrt((gk['x'] - shooter['x'])**2 + (gk['y'] - shooter['y'])**2) if gk else None
    
    dist_rows.append({
        'shot_id': s['id'],
        'success': s['success'],
        'distance': s.get('distance'),
        'nearest_other_dist': min(dists) if dists else None,
        'gk_dist': gk_dist,
        'n_others': len(others),
    })

df_dist = pd.DataFrame(dist_rows)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for success, label, color in [(0, 'Kein Tor', 'steelblue'), (1, 'Tor', 'red')]:
    sub = df_dist[df_dist['success'] == success]
    axes[0].hist(sub['nearest_other_dist'].dropna(), bins=30, alpha=0.5, label=label, color=color)
    axes[1].hist(sub['gk_dist'].dropna(), bins=30, alpha=0.5, label=label, color=color)

axes[0].set_title('Abstand nächster anderer Spieler (excl. GK)')
axes[0].set_xlabel('Abstand [m]')
axes[0].legend()

axes[1].set_title('Abstand Torwart zum Schützen')
axes[1].set_xlabel('Abstand [m]')
axes[1].legend()

plt.tight_layout()
plt.show()

print('Mittlerer GK-Abstand: Tor =', df_dist[df_dist['success']==1]['gk_dist'].mean().round(2),
      'm | Kein Tor =', df_dist[df_dist['success']==0]['gk_dist'].mean().round(2), 'm')

In [ ]:
# Heatmap: Schützenpositionen bei Toren vs Nicht-Toren (absolutes Koordinatensystem)
# Normierung: Alle Schüsse auf linke Hälfte spiegeln (shot_position_x > 0 = von links)

shooter_pos_rows = []
for s in shots_with_pos:
    if s['validated'] != 1 or s['shot_position_x'] is None:
        continue
    shooter = next((p for p in s['player_positions'] if p['player_id'] == s['player_id']), None)
    if not shooter:
        continue
    # Normiere auf Angriffsrichtung: shot_position_x sollte immer > 0 zeigen
    # shot_position_x < 0 bedeutet Schuss von links (in Richtung des Tors bei x=0)
    # Normieren: absolut x so spiegeln dass Schuss immer Richtung x=40 geht
    # player_pos.x - 20 = shot_pos_x → wenn shot_pos_x < 0, Schuss Richtung x<20
    norm_x = shooter['x'] if s['shot_position_x'] < 0 else 40 - shooter['x']
    norm_y = shooter['y'] if s['shot_position_x'] < 0 else 20 - shooter['y']
    shooter_pos_rows.append({'x': norm_x, 'y': norm_y, 'success': s['success']})

df_sp = pd.DataFrame(shooter_pos_rows)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
titles = ['Kein Tor', 'Tor']

for i, (success_val, title) in enumerate(zip([0, 1], titles)):
    ax = axes[i]
    sub = df_sp[df_sp['success'] == success_val]
    
    # Feld
    field = patches.Rectangle((0, 0), 40, 20, linewidth=1.5,
                                edgecolor='green', facecolor='#f0fff0')
    ax.add_patch(field)
    ax.axvline(20, color='green', linewidth=0.8, linestyle='--', alpha=0.4)
    # Tor bei x=0
    goal = patches.Rectangle((-0.5, 8.5), 0.5, 3, linewidth=1.5,
                               edgecolor='black', facecolor='white')
    ax.add_patch(goal)
    
    h = ax.hexbin(sub['x'], sub['y'], gridsize=20, cmap='YlOrRd', mincnt=1)
    plt.colorbar(h, ax=ax, label='Anzahl Shots')
    ax.set_xlim(-1, 41)
    ax.set_ylim(-1, 21)
    ax.set_xlabel('x [m]')
    ax.set_ylabel('y [m]')
    ax.set_title(f'Schützenpositionen – {title} (n={len(sub)})')
    ax.set_aspect('equal')
    ax.annotate('Tor →', xy=(0, 10), fontsize=9, ha='right', color='black')

plt.suptitle('Normiert: Tor immer bei x=0', fontsize=12)
plt.tight_layout()
plt.show()

## 7. Verfügbarkeit historischer Daten (DB-Check)

In [ ]:
# Wie viele Spiele sind in der DB? Für welche fehlt das 4. Fixture?
df_matches = con.execute("""
    SELECT fixture_id, match_name, start_time, team_name_home, team_name_away
    FROM main.matches_normalized
    ORDER BY start_time
""").df()

print(f'Gesamt Spiele in DB: {len(df_matches)}')
print()
print('Unsere 4 Testspiele in DB?')
check_names = [
    ('Bergischer HC', 'Flensburg', '2025-09-21'),
    ('Lemgo', 'Rhein-Neckar', '2025-10-16'),
    ('Lemgo', 'Hannover', '2025-11-09'),
    ('Magdeburg', 'Füchse', '2026-03-28'),
]
for team_a, team_b, date in check_names:
    match = df_matches[
        (df_matches['match_name'].str.contains(team_a, na=False) |
         df_matches['team_name_home'].str.contains(team_a, na=False)) &
        df_matches['start_time'].str.startswith(date)
    ]
    status = '✓ in DB' if len(match) > 0 else '✗ FEHLT'
    print(f'  {date} {team_a} vs {team_b}: {status}')
    if len(match) > 0:
        print(f'    fixture_id: {match.iloc[0]["fixture_id"]}')

In [ ]:
# Timeline der verfügbaren Spiele in DB
df_matches['start_date'] = pd.to_datetime(df_matches['start_time'].str[:10])
monthly = df_matches.groupby(df_matches['start_date'].dt.to_period('M')).size()

fig, ax = plt.subplots(figsize=(12, 3))
monthly.plot(kind='bar', ax=ax)
ax.set_title('Verfügbare Spiele in DB nach Monat')
ax.set_xlabel('')
ax.set_ylabel('Anzahl Spiele')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(f'Datumbereich: {df_matches["start_date"].min().date()} bis {df_matches["start_date"].max().date()}')

## 8. Zusammenfassung: Fragen für Christian (Termin 14. April)

In [ ]:
questions = """
FRAGEN FÜR DEN TERMIN AM 14. APRIL MIT CHRISTIAN HÜLSEMEYER
=============================================================

Bereits durch eigene Analyse / Pipeline-Docs geklärt — nur zur Bestätigung:

✓  league_id in player_positions = Sportradar Player-ID
     Verifiziert: stimmt zu >99% mit top-level league_id überein.
     Gilt das garantiert für alle Spieler und Saisons?

✓  Dateiname-Zahl (3084, 3123, 3146) = Kinexon session_id
     Exakter Match mit unserer matches_normalized-Tabelle.
     Wird session_id auch im API-Response-Body mitgeliefert?

✓  shot_type=1 = Tor, shot_type=0 = Kein Tor
     Aus DB-Cross-Tab: shot_type=1 → 100% Torquote, shot_type=0 → 0%.
     Was bedeuten shot_type=3 (n=18) und shot_type=4 (n=9)?

✓  player_positions x/y = absolutes Feldkoordinatensystem (0–40m × 0–20m)
     Sieht für alle 4 Spiele konsistent aus. Gilt das für alle Arenen garantiert?

✓  shot_position_x = player_x − 20 (Mittellinie als Nullpunkt), std=0.07m
     Bestätigt?

✓  Timestamp-Verlässlichkeit: Kinexon nutzt shared hardware clock für alle Sensoren.
     Ist player_positions ein Snapshot exakt zum Wurfzeitpunkt (timestamp_ms des Shots)?
     Oder gibt es einen systematischen Offset (z.B. letzter vollständiger Frame davor)?

------------------------------------------------------------------------

Echte offene Fragen:

1.  shot_position_y Vorzeichen ist arena-abhängig!
    Lemgo: shot_pos_y =  (player_y − 10)
    Bergischer HC + Magdeburg: shot_pos_y = −(player_y − 10)
    → Ist das bekannt? Wird es in der neuen API normiert (immer gleiche Richtung)?
    → Falls nicht: wir brauchen einen Arena-Flag oder nutzen player_positions direkt.

2.  last_group Encoding: Was bedeuten 1, 2, 3?
    → last_group=3: 67 Shots, davon 0 Tore, game_clock oft ~30:00 oder 54:54.
      Verlängerung? Zeitraum unmittelbar vor/nach Halbzeit? Freigewürfe?

3.  Warum fehlt der Torwart manchmal in player_positions?
    → 8% der Schüsse mit goalkeeper_id haben den GK nicht im Positions-Array.
      Auf der Bank? Verletzungsunterbrechung? Tracking-Lücke?

4.  Historische Daten für Modelltraining:
    → Ab 20. April neue API — gilt das rückwirkend für Saison 24/25?
    → Für Training brauchen wir möglichst viele Spiele mit player_positions.
    → Wie viele Spiele können wir rückwirkend anreichern lassen?
"""

print(questions)

In [ ]:
con.close()